# HomeVision AI — California Housing Linear Regression
This notebook covers loading, EDA, train/test split, LinearRegression training, and evaluation.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


In [ ]:
# Try to load from sklearn; fall back to synthetic data if unavailable
try:
    data = fetch_california_housing(as_frame=True)
    df = data.frame.copy()
    df.rename(columns={'MedHouseVal': 'target_price_100k'}, inplace=True)
    print('Loaded real California Housing dataset.')
except Exception as exc:
    print(f'Could not download dataset ({exc}). Using synthetic data.')
    rng = np.random.default_rng(42)
    n = 20640
    med_inc = rng.lognormal(1.6, 0.5, n).clip(0.5, 15)
    house_age = rng.uniform(1, 52, n)
    ave_rooms = rng.lognormal(1.6, 0.45, n).clip(1, 15)
    ave_bedrms = (ave_rooms / rng.uniform(3.5, 5.5, n)).clip(0.5, 5)
    population = rng.lognormal(7.0, 0.9, n).clip(3, 35682)
    ave_occup = rng.lognormal(1.1, 0.35, n).clip(0.5, 10)
    latitude = rng.uniform(32.5, 42.0, n)
    longitude = rng.uniform(-124.5, -114.5, n)
    noise = rng.normal(0, 0.3, n)
    target = (0.42*med_inc + 0.005*house_age + 0.12*ave_rooms - 0.05*ave_bedrms
              - 0.00003*population - 0.05*ave_occup
              - 0.12*np.abs(latitude-34.0) - 0.08*np.abs(longitude+118.0)
              + noise + 1.2).clip(0.15, 5.0)
    df = pd.DataFrame({'MedInc': med_inc, 'HouseAge': house_age, 'AveRooms': ave_rooms,
                        'AveBedrms': ave_bedrms, 'Population': population, 'AveOccup': ave_occup,
                        'Latitude': latitude, 'Longitude': longitude, 'target_price_100k': target})

print(df.shape)
df.head()


In [ ]:
df.describe()


In [ ]:
df.isna().sum()


In [ ]:
corr = df.corr(numeric_only=True)
corr['MedHouseVal'].sort_values(ascending=False)


In [ ]:
X = df.drop(columns=['MedHouseVal'])
y = df['MedHouseVal']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
model = LinearRegression()
model.fit(X_train, y_train)
preds = model.predict(X_test)
mae = mean_absolute_error(y_test, preds)
rmse = np.sqrt(mean_squared_error(y_test, preds))
r2 = r2_score(y_test, preds)
mae, rmse, r2


In [ ]:
plt.figure(figsize=(8,6))
plt.scatter(y_test, preds, alpha=0.35)
plt.xlabel('Actual')
plt.ylabel('Predicted')
plt.title('Predicted vs Actual')
plt.show()


## Next Step
For real image-based valuation, use a dataset containing house images and prices, then train a CNN/ViT image branch and fuse it with tabular features.